# 02 · Raw

Converte os parquets da landing em tabelas Delta gerenciadas.

Contrato desta camada:

| Regra | Motivo |
|---|---|
| Nenhuma linha descartada | A raw precisa ser um espelho fiel da origem |
| Nomes normalizados para snake_case | A TLC publica `airport_fee` em janeiro e `Airport_fee` depois |
| Todas as colunas de origem em `string` | `airport_fee` muda de `int64` para `double` entre janeiro e fevereiro; ler tudo como texto evita que a ingestão quebre quando a origem mudar de tipo de novo |
| Colunas `_source_file` e `_ingested_at` | Linhagem: dá para rastrear qualquer linha até o arquivo de origem |
| Partição `ref_year`/`ref_month` | Mês de **competência do arquivo**, deliberadamente separado do mês do evento |

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# Coloca a raiz do repositório no sys.path.
# Sobe diretórios até achar a pasta `src`, de modo que o notebook funcione
# tanto em Git Folders quanto em Workspace Files, sem caminho hardcoded.
import os
import sys

_root = os.getcwd()
while _root != "/" and not os.path.isdir(os.path.join(_root, "src")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

In [0]:
from src import config
from src.processing.landing_to_raw import LandingToRawProcessor

processor = LandingToRawProcessor(spark)

for trip_type in config.TRIP_TYPES:
    processor.process(trip_type)

## Conferência: contagem por partição

In [0]:
%sql
SELECT 'yellow' AS trip_type, ref_year, ref_month, COUNT(*) AS linhas
FROM ifood_case.raw.ny_taxi_trip_yellow GROUP BY ALL
UNION ALL
SELECT 'green', ref_year, ref_month, COUNT(*)
FROM ifood_case.raw.ny_taxi_trip_green GROUP BY ALL
ORDER BY trip_type, ref_year, ref_month;

In [0]:
%sql
-- Prova de que a linhagem funciona: cada partição vem de um arquivo só.
SELECT ref_year, ref_month, _source_file, COUNT(*) AS linhas
FROM ifood_case.raw.ny_taxi_trip_yellow
GROUP BY ALL ORDER BY ref_year, ref_month;